In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Sestavení Circuit

<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme používat tyto nebo novější verze.

```
qiskit[all]~=2.3.0
```
</details>
Tato stránka se podrobněji věnuje třídě [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) v Qiskit SDK, včetně některých pokročilejších metod, které můžeš použít k vytváření kvantových Circuit.
## Co je kvantový Circuit?
Jednoduchý kvantový Circuit je kolekce Qubitů a seznam instrukcí, které na tyto Qubity působí. Pro ilustraci následující buňka vytvoří nový Circuit se dvěma novými Qubity a poté zobrazí atribut [`qubits`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qubits) tohoto Circuit, což je seznam [`Qubitů`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Qubit) seřazených od nejméně významného bitu $q_0$ po nejvýznamnější bit $q_n$.

In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.qubits

[<Qubit register=(2, "q"), index=0>, <Qubit register=(2, "q"), index=1>]

Multiple `QuantumRegister` and `ClassicalRegister` objects can be combined to create a circuit. Every [`QuantumRegister`](/docs/api/qiskit/circuit#qiskit.circuit.QuantumRegister) and [`ClassicalRegister`](/docs/api/qiskit/circuit#qiskit.circuit.ClassicalRegister) can also be named.

In [2]:
from qiskit.circuit import QuantumRegister, ClassicalRegister

qr1 = QuantumRegister(2, "qreg1")  # Create a QuantumRegister with 2 qubits
qr2 = QuantumRegister(1, "qreg2")  # Create a QuantumRegister with 1 qubit
cr1 = ClassicalRegister(3, "creg1")  # Create a ClassicalRegister with 3 cbits

combined_circ = QuantumCircuit(
    qr1, qr2, cr1
)  # Create a quantum circuit with 2 QuantumRegisters and 1 ClassicalRegister
combined_circ.qubits

[<Qubit register=(2, "qreg1"), index=0>,
 <Qubit register=(2, "qreg1"), index=1>,
 <Qubit register=(1, "qreg2"), index=0>]

Více objektů `QuantumRegister` a `ClassicalRegister` lze zkombinovat a vytvořit tak Circuit. Každý [`QuantumRegister`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.QuantumRegister) a [`ClassicalRegister`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.ClassicalRegister) lze také pojmenovat.

In [3]:
desired_qubit = qr2[0]  # Qubit 0 of register 'qreg2'

print("Index:", combined_circ.find_bit(desired_qubit).index)
print("Register:", combined_circ.find_bit(desired_qubit).registers)

Index: 2
Register: [(QuantumRegister(1, 'qreg2'), 0)]


Adding an instruction to the circuit appends the instruction to the circuit's [`data`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#data) attribute. The following cell output shows `data` is a list of [`CircuitInstruction`](/docs/api/qiskit/qiskit.circuit.CircuitInstruction) objects, each of which has an `operation` attribute, and a `qubits` attribute.

In [4]:
qc.x(0)  # Add X-gate to qubit 0
qc.data

[CircuitInstruction(operation=Instruction(name='x', num_qubits=1, num_clbits=0, params=[]), qubits=(<Qubit register=(2, "q"), index=0>,), clbits=())]

Index a registr Qubitu lze zjistit pomocí metody [`find_bit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.find_bit) Circuit a jejích atributů.

In [5]:
qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/43a57258-3e33-4071-8a48-2bf127c8a5be-0.svg" alt="Output of the previous code cell" />

Circuit instruction objects can contain "definition" circuits that describe the instruction in terms of more fundamental instructions. For example, the [X-gate](/docs/api/qiskit/qiskit.circuit.library.XGate) is defined as a specific case of the [U3-gate](/docs/api/qiskit/qiskit.circuit.library.U3Gate), a more general single-qubit gate.

In [6]:
# Draw definition circuit of 0th instruction in `qc`
qc.data[0].operation.definition.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/653e2427-e301-4d2f-84de-1959185ace8e-0.svg" alt="Output of the previous code cell" />

Přidání instrukce do Circuit připojí tuto instrukci k atributu [`data`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#data) daného Circuit. Výstup následující buňky ukazuje, že `data` je seznam objektů [`CircuitInstruction`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.CircuitInstruction), z nichž každý má atribut `operation` a atribut `qubits`.

In [7]:
from qiskit.circuit.library import HGate

qc = QuantumCircuit(1)
qc.append(
    HGate(),  # New HGate instruction
    [0],  # Apply to qubit 0
)
qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/66813cae-9841-47ea-96b7-8fd7b82e9759-0.svg" alt="Output of the previous code cell" />

To combine two circuits, use the [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method. This accepts another [`QuantumCircuit`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit) and an optional list of qubit mappings.

<Admonition type="note">
    The [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method returns a new circuit and does **not** mutate either circuit it acts on. To mutate the circuit on which you're calling the [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method, use the argument `inplace=True`.
</Admonition>

In [8]:
qc_a = QuantumCircuit(4)
qc_a.x(0)

qc_b = QuantumCircuit(2, name="qc_b")
qc_b.y(0)
qc_b.z(1)

# compose qubits (0, 1) of qc_a to qubits (1, 3) of qc_b respectively
combined = qc_a.compose(qc_b, qubits=[1, 3])
combined.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/29152dfa-2275-4bc4-aadb-82185b9e0e86-0.svg" alt="Output of the previous code cell" />

Nejjednodušší způsob, jak tyto informace zobrazit, je metoda [`draw`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#draw), která vrátí vizualizaci Circuit. Různé způsoby zobrazení kvantových Circuit najdeš v části [Vizualizace Circuit](./visualize-circuits).

In [9]:
inst = qc_b.to_instruction()
qc_a.append(inst, [1, 3])
qc_a.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/81b682dd-45cb-4492-809e-d9e8ebbf5600-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/43a57258-3e33-4071-8a48-2bf127c8a5be-0.svg)

Objekty instrukcí Circuit mohou obsahovat „definiční" Circuit, které popisují danou instrukci pomocí základnějších instrukcí. Například [X-Gate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.XGate) je definován jako speciální případ [U3-Gate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.U3Gate), obecnějšího jednoqubitového Gate.

In [10]:
gate = qc_b.to_gate().control()
qc_a.append(gate, [0, 1, 3])
qc_a.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/ed362e64-d6a4-4dfd-a5cf-5e6bdc7a81b5-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/653e2427-e301-4d2f-84de-1959185ace8e-0.svg)

Instrukce a Circuit jsou si podobné v tom, že obě popisují operace na bitech a Qubitech, ale mají různé účely:

- Instrukce jsou považovány za neměnné a jejich metody obvykle vrátí nové instrukce (aniž by mutovaly původní objekt).
- Circuit jsou navrženy tak, aby se sestavovaly postupně přes mnoho řádků kódu, a metody [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) často mutují stávající objekt.
### Co je hloubka Circuit?
[Hloubka (depth())](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.depth) kvantového Circuit je míra počtu „vrstev" kvantových Gate prováděných paralelně, které jsou potřeba k dokončení výpočtu definovaného tímto Circuit. Protože implementace kvantových Gate trvá určitou dobu, hloubka Circuit přibližně odpovídá době, po kterou kvantový počítač Circuit provádí. Hloubka Circuit je proto jednou z důležitých veličin používaných k posouzení, zda lze daný kvantový Circuit na zařízení spustit.

Zbytek této stránky ukazuje, jak kvantové Circuit manipulovat.
## Sestavení Circuit
Metody jako [`QuantumCircuit.h`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#h) a [`QuantumCircuit.cx`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#cx) přidávají do Circuit konkrétní instrukce. Pro obecnější přidávání instrukcí do Circuit použij metodu [`append`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#append). Ta přijímá instrukci a seznam Qubitů, na které se má instrukce aplikovat. Seznam podporovaných instrukcí najdeš v [dokumentaci API Circuit Library](https://docs.quantum.ibm.com/api/qiskit/circuit_library).

In [11]:
qc_a.decompose().draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/3c0633db-929b-4428-a888-7a3d493bd6dd-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/66813cae-9841-47ea-96b7-8fd7b82e9759-0.svg)

Pro spojení dvou Circuit použij metodu [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose). Ta přijímá další [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) a volitelný seznam mapování Qubitů.

> **Note:** Metoda [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose) vrátí nový Circuit a **nemutuje** žádný z Circuit, na které působí. Chceš-li mutovat Circuit, na kterém metodu [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose) voláš, použij argument `inplace=True`.

In [12]:
qc1 = QuantumCircuit(2, 2)
qc1.measure(0, 1)
qc1.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/0cdb2273-0.svg" alt="Output of the previous code cell" />

In [13]:
qc2 = QuantumCircuit(2)
qc2.measure_all()
qc2.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/6f33698c-0.svg" alt="Output of the previous code cell" />

In [14]:
qc3 = QuantumCircuit(2)
qc3.x(1)
qc3.measure_active()
qc3.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/ca3f225f-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/ed362e64-d6a4-4dfd-a5cf-5e6bdc7a81b5-0.svg)

Chceš-li vidět, co se děje, můžeš použít metodu [`decompose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#decompose), která každou instrukci rozloží na její definici.

> **Note:** Metoda [`decompose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#decompose) vrátí nový Circuit a **nemutuje** Circuit, na který působí.

In [15]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.circuit import Parameter

angle = Parameter("angle")  # undefined number

# Create and optimize circuit once
qc = QuantumCircuit(1)
qc.rx(angle, 0)
qc = generate_preset_pass_manager(
    optimization_level=3, basis_gates=["u", "cx"]
).run(qc)

qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/a580552c-d585-4047-99f0-32aafd06e4f3-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/3c0633db-929b-4428-a888-7a3d493bd6dd-0.svg)

<span id="measure-qubits"></span>
## Měření Qubitů
Měření slouží k vzorkování stavů jednotlivých Qubitů a přenosu výsledků do klasického registru. Pokud odesíláš Circuit do primitiva [Sampler](./primitives#sampler), měření jsou povinná. Circuit odesílané do primitiva [Estimator](./primitives#estimator) naopak měření nesmějí obsahovat.

Qubity lze měřit třemi metodami: [`measure`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.measure), [`measure_all`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) a [`measure_active`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_active). Jak vizualizovat naměřené výsledky se dozvíš na stránce [Vizualizace výsledků](./visualize-results).

1. `QuantumCircuit.measure` : měří každý Qubit uvedený v prvním argumentu na klasický bit zadaný jako druhý argument. Tato metoda umožňuje plnou kontrolu nad tím, kam se výsledek měření uloží.

2. `QuantumCircuit.measure_all` : nepřijímá žádný argument a lze ji použít pro kvantové Circuit bez předem definovaných klasických bitů. Vytvoří klasické dráty a ukládá výsledky měření v pořadí. Například měření Qubitu $q_i$ se uloží do cbitu $meas_i$). Před měřením také přidá bariéru.

3. `QuantumCircuit.measure_active` : podobná metodě `measure_all`, ale měří pouze Qubity, které mají operace.

In [16]:
circuits = []
for value in range(100):
    circuits.append(qc.assign_parameters({angle: value}))

circuits[0].draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/85af6231-921a-4130-99d3-f6998f761df8-0.svg" alt="Output of the previous code cell" />

You can find a list of a circuit's undefined parameters in its `parameters` attribute.

In [17]:
qc.parameters

ParameterView([Parameter(angle)])

### Change a parameter's name

By default, parameter names for a parameterized circuit are prefixed by `x`- for example, `x[0]`. You can change the names after they are defined, as shown in the following example.

In [18]:
from qiskit.circuit.library import z_feature_map
from qiskit.circuit import ParameterVector

# Define a parameterized circuit with default names
# For example, x[0]
circuit = z_feature_map(2)

# Set new parameter names
# They will now be prefixed by `hi` instead
# For example, hi[0]
training_params = ParameterVector("hi", 2)

# Assign parameter names to the quantum circuit
circuit = circuit.assign_parameters(parameters=training_params)

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/ca3f225f-0.svg)

## Parametrizované Circuit

Mnoho krátkodobých kvantových algoritmů zahrnuje provádění mnoha variací kvantového Circuit. Protože sestavování a optimalizace rozsáhlých Circuit může být výpočetně nákladné, Qiskit podporuje **parametrizované** Circuit. Tyto Circuit mají nedefinované parametry, jejichž hodnoty nemusí být určeny až do okamžiku těsně před spuštěním Circuit. To ti umožňuje přesunout sestavení a optimalizaci Circuit mimo hlavní programovou smyčku. Následující buňka vytvoří a zobrazí parametrizovaný Circuit.